# Worksheet 03 Solution

This notebook completes **Worksheet 3: Modelling the Neuron**.

It covers:

- MCP neurons for `AND` and `OR`
- written answers for the MCP questions
- perceptron learning for `0 vs 1` digit classification
- perceptron learning for `3 vs 5` digit classification
- visualization of misclassified images and conclusions


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


## Task 1: MCP Neurons

We first implement simple MCP neurons for the Boolean `AND` and `OR` functions with two inputs.


In [ ]:
def MCP_Neurons_AND(X1, X2, T):
    """
    Implements AND logic with a two-input MCP neuron.
    """
    assert len(X1) == len(X2)

    state_neuron = []
    for x1, x2 in zip(X1, X2):
        total_input = x1 + x2
        state_neuron.append(1 if total_input >= T else 0)

    return state_neuron


X1 = [0, 0, 1, 1]
X2 = [0, 1, 0, 1]
T = 2

result_and = MCP_Neurons_AND(X1, X2, T)
print(f"Output of AND gate for inputs {X1} and {X2} with threshold {T}: {result_and}")


In [ ]:
def MCP_Neurons_OR(X1, X2, T):
    """
    Implements OR logic with a two-input MCP neuron.
    """
    assert len(X1) == len(X2)

    state_neuron = []
    for x1, x2 in zip(X1, X2):
        total_input = x1 + x2
        state_neuron.append(1 if total_input >= T else 0)

    return state_neuron


X1 = [0, 0, 1, 1]
X2 = [0, 1, 0, 1]
T = 1

result_or = MCP_Neurons_OR(X1, X2, T)
print(f"Output of OR gate for inputs {X1} and {X2} with threshold {T}: {result_or}")


### Answer: Question 1

**List out all the limitations of MCP neurons.**

MCP neurons have several important limitations:

- They use a fixed hand-crafted threshold instead of learning from data.
- They work best with binary-style inputs and outputs, so they are too simple for richer data.
- A single MCP neuron can only represent linearly separable decision boundaries.
- They cannot solve XOR with a single threshold unit.
- They do not learn weights automatically, so they cannot adapt when the data changes.
- They are highly simplified and do not represent real biological neurons accurately.
- They do not provide probabilistic outputs or confidence values.
- They are not robust to noisy, ambiguous, or high-dimensional real-world inputs.


### Answer: Question 2

**Think if you can develop a logic to solve XOR using MCP Neuron.**

If we are allowed to write a direct `if-else` rule such as:

- output `1` when the two inputs are different
- output `0` when the two inputs are the same

then we can reproduce XOR behavior procedurally.

However, a **single MCP neuron** still cannot solve XOR, because XOR is **not linearly separable**. To solve XOR using MCP-style units, we need **multiple neurons arranged in layers**, not one threshold gate alone.


## Task 2: Perceptron Algorithm for 0 vs 1 Classification

We now implement a perceptron from scratch and apply it to the MNIST `0 vs 1` dataset.


In [ ]:
def resolve_dataset_path(filename):
    candidates = [
        Path(filename),
        Path.cwd() / filename,
        Path("/Users/r5sman/Downloads") / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find dataset file: {filename}")


def load_binary_digit_dataset(filename):
    path = resolve_dataset_path(filename)
    df = pd.read_csv(path)

    X = df.drop(columns=["label"]).to_numpy(dtype=np.float32) / 255.0
    y_original = df["label"].to_numpy()
    class_values = sorted(np.unique(y_original))

    label_map = {class_values[0]: 0, class_values[1]: 1}
    y_binary = np.vectorize(label_map.get)(y_original).astype(np.int64)

    return X, y_binary, y_original, class_values, path


def show_class_samples(X, y_original, class_values, title):
    fig, axes = plt.subplots(2, 5, figsize=(10, 5))

    top_class = class_values[0]
    bottom_class = class_values[1]

    images_top = X[y_original == top_class]
    images_bottom = X[y_original == bottom_class]

    if len(images_top) < 5 or len(images_bottom) < 5:
        print("Error: Not enough images to display 5 samples per class.")
        return

    for i in range(5):
        axes[0, i].imshow(images_top[i].reshape(28, 28), cmap="gray")
        axes[0, i].set_title(f"Label: {top_class}")
        axes[0, i].axis("off")

        axes[1, i].imshow(images_bottom[i].reshape(28, 28), cmap="gray")
        axes[1, i].set_title(f"Label: {bottom_class}")
        axes[1, i].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def initialize_parameters(n_features):
    weights = np.zeros(n_features, dtype=np.float32)
    bias = 0.0
    return weights, bias


def decision_function(X, weights, bias):
    """
    Compute predicted labels using the perceptron step function.
    """
    predictions = np.dot(X, weights) + bias
    y_pred_all = np.where(predictions >= 0, 1, 0)
    return y_pred_all


def train_perceptron(X, y, weights, bias, learning_rate=0.1, epochs=100, verbose=True):
    """
    Train a perceptron using the perceptron learning algorithm.
    """
    weights = weights.copy().astype(np.float32)
    bias = float(bias)
    history = []

    for epoch in range(epochs):
        errors = 0

        for i in range(X.shape[0]):
            output = np.dot(X[i], weights) + bias
            prediction = 1 if output >= 0 else 0
            error = y[i] - prediction

            if error != 0:
                weights += learning_rate * error * X[i]
                bias += learning_rate * error
                errors += 1

        y_pred_epoch = decision_function(X, weights, bias)
        accuracy = np.mean(y_pred_epoch == y)
        history.append(
            {
                "epoch": epoch + 1,
                "errors": errors,
                "accuracy": accuracy,
            }
        )

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0 or errors == 0):
            print(
                f"Epoch {epoch + 1:>3}: errors = {errors:>4}, "
                f"accuracy = {accuracy:.4f}"
            )

        if errors == 0:
            break

    accuracy = history[-1]["accuracy"]
    return weights, bias, accuracy, history


def plot_training_history(history, title):
    epochs = [item["epoch"] for item in history]
    errors = [item["errors"] for item in history]
    accuracies = [item["accuracy"] for item in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, errors, color="crimson")
    axes[0].set_title("Misclassifications per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Errors")

    axes[1].plot(epochs, accuracies, color="navy")
    axes[1].set_title("Accuracy per Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1.02)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def visualize_misclassified_images(X, y_true_original, y_pred_binary, class_values, title, max_images=10):
    predicted_original = np.array([class_values[idx] for idx in y_pred_binary])
    misclassified_idx = np.where(predicted_original != y_true_original)[0]

    if len(misclassified_idx) == 0:
        print("All images were correctly classified!")
        return misclassified_idx

    fig, axes = plt.subplots(2, 5, figsize=(10, 5))
    for ax, idx in zip(axes.flat, misclassified_idx[:max_images]):
        ax.imshow(X[idx].reshape(28, 28), cmap="gray")
        ax.set_title(f"Pred: {predicted_original[idx]}, True: {y_true_original[idx]}")
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    return misclassified_idx


In [ ]:
X_01, y_01, y_01_original, class_values_01, dataset_path_01 = load_binary_digit_dataset("mnist_0_and_1.csv")

print("Dataset path:", dataset_path_01)
print("Feature matrix shape:", X_01.shape)
print("Label vector shape:", y_01.shape)


### Answer: Question 1

The shape of `X` represents the structure of the input feature matrix.

- The first dimension is the number of images or samples.
- The second dimension is the number of features per image.

For MNIST, each image is `28 x 28 = 784` pixels, so each row of `X` stores one flattened image.

### Answer: Question 2

The worksheet repeats the same question text, but the second one is best interpreted as the shape of `y`.

The shape of `y` represents the label vector:

- it has one label for each image in `X`
- so its length must match the number of rows in `X`


In [ ]:
show_class_samples(
    X_01,
    y_01_original,
    class_values_01,
    title="First 5 Images of 0 and 1 from MNIST Subset",
)


In [ ]:
weights_01, bias_01 = initialize_parameters(X_01.shape[1])
learning_rate_01 = 0.1
epochs_01 = 100

print("Weights shape:", weights_01.shape)
print("Initial bias:", bias_01)


### Answer: Question 3

The `weights` array stores one weight for each pixel feature. Since every image has 784 pixels, the perceptron learns 784 weights. These weights tell the model how strongly each pixel contributes to the decision boundary.

### Answer: Question 4

We initialize the weights to zero as a simple and neutral starting point. For a perceptron, this is acceptable because the weights are updated whenever a sample is misclassified. The main effect is that the first few updates are driven entirely by the earliest mistakes in the dataset, but the model still learns through repeated corrections.


### Answer: Question 5

The line `output = np.dot(X[i], weights) + bias` computes the **net input** of the perceptron for one sample. It combines the weighted pixel values and the bias to measure how strongly the sample supports one class versus the other.

### Answer: Question 6

When the prediction is wrong, the perceptron updates both the weights and the bias:

- `weights = weights + learning_rate * error * X[i]`
- `bias = bias + learning_rate * error`

This shifts the decision boundary so the current sample is more likely to be classified correctly in later epochs.

### Answer: Question 7

Final accuracy is important because it summarizes how well the perceptron separates the dataset after training. For `0 vs 1`, we expect the accuracy to be extremely high, often even perfect on the training set, because those two digits are relatively easy to separate.


In [ ]:
weights_01, bias_01, accuracy_01, history_01 = train_perceptron(
    X_01,
    y_01,
    weights_01,
    bias_01,
    learning_rate=learning_rate_01,
    epochs=epochs_01,
    verbose=True,
)

print("The Final Accuracy is:", accuracy_01)


In [ ]:
plot_training_history(history_01, "Perceptron Training History for 0 vs 1")

y_pred_01 = decision_function(X_01, weights_01, bias_01)
final_accuracy_01 = np.mean(y_pred_01 == y_01)
print(f"Final Accuracy: {final_accuracy_01:.4f}")

misclassified_idx_01 = visualize_misclassified_images(
    X_01,
    y_01_original,
    y_pred_01,
    class_values_01,
    title="Misclassified Images for 0 vs 1 Classification",
)


### Answer: Question 8

`misclassified_idx` stores the indices of all samples where the predicted label does not match the true label. The code uses these indices to pull out the corresponding images and display them, so we can inspect exactly where the perceptron made mistakes.

### Answer: Question 9

If the output is **"All images were correctly classified!"**, it means the trained perceptron made no mistakes on the dataset it was evaluated on. In other words, the data was perfectly separated by the learned linear decision boundary.


## Task 3: Perceptron Algorithm for 3 vs 5 Classification

We now repeat the same perceptron pipeline for the digits `3` and `5`.


In [ ]:
X_35, y_35, y_35_original, class_values_35, dataset_path_35 = load_binary_digit_dataset("mnist_3_and_5.csv")

print("Dataset path:", dataset_path_35)
print("Feature matrix shape:", X_35.shape)
print("Label vector shape:", y_35.shape)

show_class_samples(
    X_35,
    y_35_original,
    class_values_35,
    title="First 5 Images of 3 and 5 from MNIST Subset",
)


In [ ]:
weights_35, bias_35 = initialize_parameters(X_35.shape[1])
learning_rate_35 = 0.1
epochs_35 = 100

weights_35, bias_35, accuracy_35, history_35 = train_perceptron(
    X_35,
    y_35,
    weights_35,
    bias_35,
    learning_rate=learning_rate_35,
    epochs=epochs_35,
    verbose=True,
)

print("The Final Accuracy for 3 vs 5 is:", accuracy_35)


In [ ]:
plot_training_history(history_35, "Perceptron Training History for 3 vs 5")

y_pred_35 = decision_function(X_35, weights_35, bias_35)
final_accuracy_35 = np.mean(y_pred_35 == y_35)
print(f"Final Accuracy: {final_accuracy_35:.4f}")

misclassified_idx_35 = visualize_misclassified_images(
    X_35,
    y_35_original,
    y_pred_35,
    class_values_35,
    title="Misclassified Images for 3 vs 5 Classification",
)


## Conclusion

The perceptron performs extremely well on `0 vs 1`, because those two classes are much easier to separate with a linear decision boundary. In our run, the model reached perfect classification on the full training dataset.

For `3 vs 5`, the perceptron still performs strongly, but usually not perfectly. These digits are visually more similar, and some examples are harder to separate with a single linear boundary. The remaining mistakes and the persistence of non-zero training errors illustrate an important limitation of the perceptron: it is a **linear classifier**, so its success depends on how linearly separable the classes are.
